# Measuring bias and variance

We train the same kind of model on 10,000 training sets and see how its predictions vary.

The design follows ESL §7.3: every training set has the same 20 inputs $x$ (the feature); only the noise in the target $y$ is new each time. We measure bias² and variance at those 20 inputs.

In [1]:
import warnings
import numpy as np
import plotly.graph_objects as go
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
warnings.simplefilter("ignore")

f = lambda x: np.sin(1.5 * x) + 0.5 * x       # the true pattern (normally unknown)
NOISE, N, SETS = 0.5, 20, 10_000
rng = np.random.default_rng(0)
X = np.sort(rng.uniform(-3, 3, (N, 1)), axis=0)   # the same 20 inputs in every training set
Y = f(X) + rng.normal(0, NOISE, (N, SETS))       # one column of noisy targets per training set
xs = np.linspace(-2.8, 2.8, 200)

In [2]:
def model(degree):
    return make_pipeline(PolynomialFeatures(degree, include_bias=False), StandardScaler(), LinearRegression())

def fits(degree):
    # LinearRegression accepts many target columns at once: one fitted model per training set
    return model(degree).fit(X, Y).predict(X)     # predictions at the 20 inputs, one column per training set

for d in range(1, 12):
    P = fits(d)
    bias2 = np.mean((P.mean(axis=1) - f(X).ravel()) ** 2)   # how far the AVERAGE model is from the truth
    var = np.mean(P.var(axis=1))                            # how much the models disagree with each other
    print(f"degree {d:2d}: bias^2 {bias2:.4f}  variance {var:.3f}  expected error {bias2 + var + NOISE**2:.3f}")

degree  1: bias^2 0.4195  variance 0.025  expected error 0.695
degree  2: bias^2 0.4147  variance 0.037  expected error 0.702
degree  3: bias^2 0.0391  variance 0.050  expected error 0.339


degree  4: bias^2 0.0386  variance 0.062  expected error 0.351
degree  5: bias^2 0.0011  variance 0.075  expected error 0.326


degree  6: bias^2 0.0007  variance 0.087  expected error 0.338


degree  7: bias^2 0.0000  variance 0.100  expected error 0.350


degree  8: bias^2 0.0000  variance 0.112  expected error 0.362


degree  9: bias^2 0.0000  variance 0.124  expected error 0.374


degree 10: bias^2 0.0000  variance 0.137  expected error 0.387
degree 11: bias^2 0.0000  variance 0.150  expected error 0.400


## Check against the theory (ESL §7.3)

For a least-squares fit with fixed inputs, the average model is the same model fitted to the noise-free targets $f(x)$, and the variance is exactly $\sigma^2 p / N$, where $p$ is the number of coefficients (degree + 1). Bias² can only fall or stay flat as the degree grows, because each larger polynomial family contains the smaller one.

In [3]:
prev = np.inf
for d in range(1, 12):
    exact_bias2 = np.mean((model(d).fit(X, f(X).ravel()).predict(X) - f(X).ravel()) ** 2)
    exact_var = NOISE**2 * (d + 1) / N
    P = fits(d)
    print(f"degree {d:2d}: bias^2 exact {exact_bias2:.6f}  measured {np.mean((P.mean(1) - f(X).ravel())**2):.6f}"
          f"   variance exact {exact_var:.4f}  measured {np.mean(P.var(1)):.4f}")
    assert exact_bias2 <= prev + 1e-12      # bias^2 never rises with the degree
    prev = exact_bias2

degree  1: bias^2 exact 0.419493  measured 0.419493   variance exact 0.0250  measured 0.0250


degree  2: bias^2 exact 0.414732  measured 0.414734   variance exact 0.0375  measured 0.0375


degree  3: bias^2 exact 0.039098  measured 0.039099   variance exact 0.0500  measured 0.0496


degree  4: bias^2 exact 0.038620  measured 0.038622   variance exact 0.0625  measured 0.0622
degree  5: bias^2 exact 0.001108  measured 0.001110   variance exact 0.0750  measured 0.0748


degree  6: bias^2 exact 0.000742  measured 0.000745   variance exact 0.0875  measured 0.0871


degree  7: bias^2 exact 0.000007  measured 0.000009   variance exact 0.1000  measured 0.0998
degree  8: bias^2 exact 0.000003  measured 0.000005   variance exact 0.1125  measured 0.1121


degree  9: bias^2 exact 0.000000  measured 0.000003   variance exact 0.1250  measured 0.1244


degree 10: bias^2 exact 0.000000  measured 0.000003   variance exact 0.1375  measured 0.1371


degree 11: bias^2 exact 0.000000  measured 0.000003   variance exact 0.1500  measured 0.1497


In [4]:
G = model(11).fit(X, Y[:, :20]).predict(xs.reshape(-1, 1))
fig = go.Figure([go.Scatter(x=xs, y=np.clip(g, -4, 4), mode="lines", opacity=0.4, showlegend=False) for g in G.T])
fig.add_scatter(x=xs, y=f(xs), mode="lines", name="truth", line=dict(color="black", dash="dash", width=3))
fig.update_layout(title="Degree 11 on 20 different training sets", yaxis_range=[-4, 4])
fig